In [ ]:
!pip install wandb pandas numpy matplotlib seaborn scikit-learn kagglehub statsmodels

In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
import kagglehub
import shutil
import os

# Set random seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)
print("All libraries imported successfully.")

In [ ]:
from key import KEY
API_KEY=KEY
wandb.login(API_KEY)

In [ ]:
path = kagglehub.dataset_download("lovishbansal123/nasa-asteroids-classification")
print(f"Dataset downloaded to: {path}")

dest = os.path.join(os.getcwd(), "dataset")
shutil.copytree(path, dest, dirs_exist_ok=True)

df_raw = pd.read_csv(os.path.join(dest, "nasa.csv"), low_memory=False)
print(f"Raw data shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Transformando o True em 1 e False em 0.
df_raw['Hazardous'] = df_raw['Hazardous'].replace({True: 1, False: 0})

df_raw.head()

# Removendo a coluna: "Equinox" e "Orbiting body"

df = df_raw.drop(['Equinox', 'Orbiting Body'], axis=1)
df.head()

In [ ]:
wandb.init(project="MLOps-Work", job_type="load_raw", name="load_raw")
artifact = wandb.Artifact("raw_data", type="dataset")
temp_path = "temp_raw.csv"
df_raw.to_csv(temp_path, index=False)
artifact.add_file(temp_path)
wandb.log_artifact(artifact)
wandb.summary["rows"] = len(df_raw)
wandb.summary["columns"] = list(df_raw.columns)
wandb.finish()
print("Raw data artifact saved to W&B.")

In [ ]:
def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates()
    print(f"Removed {before - len(df)} duplicate rows.")
    return df

def handle_missing_values(df, threshold=0.5):
    missing_frac = df.isnull().mean()
    cols_to_drop = missing_frac[missing_frac > threshold].index.tolist()
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped columns ( >{threshold*100}% missing): {cols_to_drop}")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    imputer = SimpleImputer(strategy='median')
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    cat_cols = df.select_dtypes(include=['object']).columns
    df[cat_cols] = df[cat_cols].fillna('missing')
    return df

df_clean = remove_duplicates(df_raw)
df_clean = handle_missing_values(df_clean, threshold=0.5)
print(f"Cleaned dataset shape: {df_clean.shape}")

In [ ]:
wandb.init(project="MLOps-work", job_type="clean_data", name="clean_data")
artifact = wandb.Artifact("clean_data", type="dataset")
temp_clean = "temp_clean.csv"
df_clean.to_csv(temp_clean, index=False)
artifact.add_file(temp_clean)
wandb.log_artifact(artifact)
wandb.summary["clean_rows"] = len(df_clean)
wandb.finish()
print("Cleaned data artifact saved to W&B.")

In [ ]:
TARGET = 'Hazardous'
print(f"Target variable: {TARGET}")
print("Distribution of target:")
print(df_clean[TARGET].value_counts().sort_index())